#Load dataset and preprocess text (tokenize, clean)

In [1]:
import re
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [3]:
# Load the labeled spam/ham dataset
df = pd.read_csv("spam.csv", encoding="latin-1")
df = df[["v1", "v2"]]
df.columns = ["label", "text"]
df = df.dropna(subset=["text"])

print(f"Total messages: {len(df)}")
print(df["label"].value_counts())
df.head()

Total messages: 5572
label
ham     4825
spam     747
Name: count, dtype: int64


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
def clean_and_tokenize(text: str) -> str:
    """
    Clean raw SMS text and tokenize it:
    - lowercase
    - remove URLs, numbers, punctuation
    - tokenize on word boundaries
    - remove stopwords
    Returns cleaned tokens joined back into a string, ready for the vectorizer.
    """
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # remove URLs
    text = re.sub(r"[^a-z\s]", " ", text)            # keep letters only
    tokens = text.split()                              # tokenize
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]
    return " ".join(tokens)


df["clean_text"] = df["text"].apply(clean_and_tokenize)
df[["text", "clean_text"]].head()

,text,clean_text
0,"Go until jurong point, crazy.. Available only ...",jurong point crazy available bugis great world...
1,Ok lar... Joking wif u oni...,ok lar joking wif oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,U dun say so early hor... U c already then say...,dun say early hor say
4,"Nah I don't think he goes to usf, he lives aro...",nah don think goes usf lives


In [5]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 4457, Test size: 1115


#Convert text to vectors (TF-IDF) and train Naive Bayes

In [6]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("nb", MultinomialNB()),
])

pipeline.fit(X_train, y_train)

,steps,"[('tfidf', ...), ('nb', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


#Evaluate with precision / recall / F1

In [7]:
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label="spam")
recall = recall_score(y_test, y_pred, pos_label="spam")
f1 = f1_score(y_test, y_pred, pos_label="spam")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print()
print(classification_report(y_test, y_pred))

Accuracy : 0.9632
Precision: 0.9909
Recall   : 0.7315
F1-score : 0.8417

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       0.99      0.73      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.87      0.91      1115
weighted avg       0.96      0.96      0.96      1115



In [8]:
print("Confusion matrix (rows=actual, cols=predicted) [ham, spam]:")
print(confusion_matrix(y_test, y_pred, labels=["ham", "spam"]))

Confusion matrix (rows=actual, cols=predicted) [ham, spam]:
[[965   1]
 [ 40 109]]


 #Save pipeline (vectorizer + model) to reuse for predictions

In [9]:
joblib.dump(pipeline, "spam_detector_pipeline.joblib")
print("Pipeline saved as 'spam_detector_pipeline.joblib'")

Pipeline saved as 'spam_detector_pipeline.joblib'


## Reuse the saved pipeline for new predictions

In [10]:
loaded_pipeline = joblib.load("spam_detector_pipeline.joblib")

sample_messages = [
    "Congratulations! You have WON a free ticket to Bahamas, call now to claim!!!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been suspended, click here to verify your details.",
    "Can you send me the notes from today's class?",
]

cleaned_samples = [clean_and_tokenize(m) for m in sample_messages]
predictions = loaded_pipeline.predict(cleaned_samples)

for msg, label in zip(sample_messages, predictions):
    print(f"[{label.upper():4}] {msg}")

[SPAM] Congratulations! You have WON a free ticket to Bahamas, call now to claim!!!
[HAM ] Hey, are we still meeting for lunch tomorrow?
[SPAM] URGENT: Your account has been suspended, click here to verify your details.
[HAM ] Can you send me the notes from today's class?


#Interactive Spam Checker

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

loaded_pipeline = joblib.load("spam_detector_pipeline.joblib")

text_box = widgets.Textarea(
    placeholder="type your message",
    layout=widgets.Layout(width="500px", height="80px"),
)
check_button = widgets.Button(description="Check", button_style="primary")
output_area = widgets.Output()


def on_check_clicked(b):
    output_area.clear_output()
    message = text_box.value.strip()
    with output_area:
        if not message:
            print("first write your message.")
            return
        cleaned = clean_and_tokenize(message)
        label = loaded_pipeline.predict([cleaned])[0]
        proba = loaded_pipeline.predict_proba([cleaned])[0]
        classes = loaded_pipeline.classes_
        confidence = proba[list(classes).index(label)] * 100

        if label == "spam":
            display(HTML(
                f"<h3 style='color:white;background:#e74c3c;padding:8px;border-radius:6px;'>"
                f" SPAM ({confidence:.1f}% confidence)</h3>"
            ))
        else:
            display(HTML(
                f"<h3 style='color:white;background:#27ae60;padding:8px;border-radius:6px;'>"
                f" HAM / Not Spam ({confidence:.1f}% confidence)</h3>"
            ))


check_button.on_click(on_check_clicked)

display(text_box, check_button, output_area)

Textarea(value='', layout=Layout(height='80px', width='500px'), placeholder='type your message')

Button(button_style='primary', description='Check', style=ButtonStyle())

Output()